# Chapter 13 — .gitignore & GitHub Actions
> *Tell Git what to ignore. Then automate literally everything else.* 

---

>  The `.gitignore` hall of shame — things that have been accidentally committed to public repos:
> - `node_modules/` (1.3 GB of dependencies, now on GitHub forever)
> - `.env` with production database passwords (the incident report writes itself)
> - `id_rsa` (your private SSH key, shared with the internet)
> - `.DS_Store` files in literally every Mac developer's first repo
>
> This chapter prevents all of these.

##  .gitignore — Stop Tracking Files That Shouldn't Be Tracked

```bash
# Create the .gitignore file (put it in your repo root):
touch .gitignore            # Mac / Linux / Git Bash
New-Item .gitignore         # Windows PowerShell
```

### Common patterns for every project:
```gitignore
# === Python projects ===
__pycache__/
*.pyc
*.pyo
.env                    # NEVER commit this — passwords live here
venv/
.venv/
*.egg-info/

# === Node.js / Frontend projects ===
node_modules/           # 1.3 GB. Do NOT commit. Ever. Please.
.env
dist/
build/
.next/

# === OS-generated garbage that nobody needs ===
.DS_Store               # Mac leaves this in EVERY folder like breadcrumbs
Thumbs.db               # Windows does its own thing too
desktop.ini

# === IDE files ===
.vscode/settings.json   # Your personal VS Code settings (team shares the file, not settings)
.idea/                  # JetBrains IDE folder
*.suo

# === Secrets (THE IMPORTANT ONES) ===
.env
.env.local
*.pem
*_key
id_rsa
```

```bash
# Already committed a file that should be ignored? Fix it:
git rm --cached <file>              # Remove from Git tracking, keep on your disk
git rm --cached -r node_modules/    # For folders
# Then add it to .gitignore, then commit the change
```

>  `gitignore.io` is the laziest and most effective tool.
> Go to https://gitignore.io → type: Python, Node, Windows, macOS, VS Code
> Click Generate → copy-paste the output into your `.gitignore`
> Done. You'll never manually think about what to ignore again.

##  GitHub Actions — Automate Everything That's Boring

GitHub Actions = automated workflows that run on GitHub's servers when you push/PR/etc.
Test your code. Check style. Deploy to production. Send Slack notifications. All automatic. All free (within limits).

Create the file: `.github/workflows/ci.yml` (GitHub looks for this exact path)

```yaml
name: Run Tests on Every Push
on: [push, pull_request]    # Triggers: any push OR any pull request

jobs:
  test:
    runs-on: ubuntu-latest  # GitHub provides a fresh Ubuntu machine for this
    steps:

      - name: Checkout the code
        uses: actions/checkout@v3    # Pulls your repo onto the runner

      - name: Set up Python 3.12
        uses: actions/setup-python@v4
        with:
          python-version: '3.12'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run the test suite
        run: python -m pytest

      - name: Check code style
        run: flake8 .     # Fails if code style is bad — blocks merge
```

> Every time you push, GitHub runs this:
> -  Green checkmark on your PR = tests passed, style clean
> -  Red X on your PR = something broke, fix before merge

>  GitHub Actions is the "we could hire a QA engineer,
> or we could write a 20-line YAML file that does the same job for free."
>
> Senior devs set up Actions once. Junior devs spend the next year
> puzzled why their PR is blocked. "Who keeps checking my tests?"
> The machine. The machine checks your tests. The machine never sleeps.

##  Actions — Quick Practical Examples

```yaml
# Deploy to GitHub Pages automatically on push to main:
on:
  push:
    branches: [main]
jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Deploy
        uses: peaceiris/actions-gh-pages@v3
        with:
          github_token: ${{ secrets.GITHUB_TOKEN }}
          publish_dir: ./docs
```

```yaml
# Notify Slack when deploy happens:
- name: Slack Notification
  uses: rtCamp/action-slack-notify@v2
  env:
    SLACK_WEBHOOK: ${{ secrets.SLACK_WEBHOOK }}
    SLACK_MESSAGE: "Deployed to production!"
```

>  `${{ secrets.GITHUB_TOKEN }}` is a special token GitHub provides automatically.
> For your own API keys and passwords → GitHub repo → Settings → Secrets → Actions → New secret
> NEVER hardcode credentials in YAML files. GitHub scans for that and warns you.
> The whole point is to not repeat the `.env` in public repos incident.